# ANFIS Breast Cancer Training (Paper-Style + Luật chuyên gia)

Notebook huấn luyện ANFIS bám sát paper về dữ liệu và feature, khai báo luật mờ theo kiểu `FS()` (tham khảo `test.ipynb`):
- Dữ liệu WBCD (UCI original)
- **PCA (Bảng 8 paper) → chọn 3 feature cố định: v1, v2, v3**
- **Luật mờ do chuyên gia định nghĩa qua `FS()` — THEN là *Lành tính* hoặc *Ác tính*, không phải hệ số nhân đặc trưng**
- Chia dữ liệu đúng paper: 200 train / 263 checking / 200 test
- Hàm thành viên **Gaussian** (low / medium / high), 3 mỗi input → **27 luật** (grid 3³)
- Huấn luyện hybrid 300 epoch bằng `scikit-anfis` + **lưu training loss log theo epoch**

In [17]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

from skanfis import scikit_anfis
from skanfis.fs import FS, LinguisticVariable, GaussianFuzzySet
from skanfis.experimental import RMSELoss

print("Torch:", torch.__version__)

Torch: 2.12.0+cpu


In [18]:
# 1) Load WBCD goc (UCI Breast Cancer Wisconsin Original)
uci_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data"

cols = [
    "sample_code_number",
    "clump_thickness",
    "uniformity_of_cell_size",
    "uniformity_of_cell_shape",
    "marginal_adhesion",
    "single_epithelial_cell_size",
    "bare_nuclei",
    "bland_chromatin",
    "normal_nucleoli",
    "mitoses",
    "class",
]

df = pd.read_csv(uci_url, header=None, names=cols)

# Missing value trong bo du lieu nay nam o cot bare_nuclei, duoc ma hoa bang '?'
df = df.replace("?", np.nan).dropna().copy()
df["bare_nuclei"] = df["bare_nuclei"].astype(int)

# Class goc: benign=2, malignant=4 -> doi ve 0/1
df["target"] = (df["class"] == 4).astype(int)

feature_cols = [
    "clump_thickness",
    "uniformity_of_cell_size",
    "uniformity_of_cell_shape",
    "marginal_adhesion",
    "single_epithelial_cell_size",
    "bare_nuclei",
    "bland_chromatin",
    "normal_nucleoli",
    "mitoses",
]

X = df[feature_cols].values.astype(np.float32)
y = df["target"].values.astype(np.float32)

print("Dataset shape:", X.shape)
print("Class balance (malignant rate):", y.mean().round(4))

Dataset shape: (683, 9)
Class balance (malignant rate): 0.3499


In [19]:
# 2) PCA tren 9 dac trung -> tinh % quan trong -> chon top-3 -> chia tap paper

feature_name_map = {
    "clump_thickness": "Clump Thickness",
    "uniformity_of_cell_size": "Uniformity of Cell Size",
    "uniformity_of_cell_shape": "Uniformity of Cell Shape",
    "marginal_adhesion": "Marginal Adhesion",
    "single_epithelial_cell_size": "Single Epithelial Cell Size",
    "bare_nuclei": "Bare Nuclei",
    "bland_chromatin": "Bland Chromatin",
    "normal_nucleoli": "Normal Nucleoli",
    "mitoses": "Mitoses",
}

# Loai ngoai lai bang khoang cach Euclidean tren 9 dac trung goc (WBCD scale 1-10)
OUTLIER_STD_MULTIPLIER = 2.5
y_all = y.astype(int)

data_center = X.mean(axis=0)
euclidean_distances = np.linalg.norm(X - data_center, axis=1)
outlier_threshold = euclidean_distances.mean() + OUTLIER_STD_MULTIPLIER * euclidean_distances.std()
outlier_mask = euclidean_distances > outlier_threshold
keep_mask = ~outlier_mask

X_clean = X[keep_mask]
y_clean = y_all[keep_mask]
df_clean = df.iloc[keep_mask].reset_index(drop=True)

outlier_dropped_df = df.iloc[outlier_mask][["sample_code_number", "class"]].copy()
outlier_dropped_df["euclidean_distance"] = euclidean_distances[outlier_mask]
outlier_dropped_df["outlier_threshold"] = outlier_threshold
outlier_dropped_df = outlier_dropped_df.sort_values("euclidean_distance", ascending=False).reset_index(drop=True)

print(f"\nOutlier Euclidean (mean + {OUTLIER_STD_MULTIPLIER}*std): {int(outlier_mask.sum())} mau")
print("Nguong outlier:", round(float(outlier_threshold), 4))
print("Con lai sau loai outlier:", len(X_clean))

# Normalize toan bo 9 dac trung (paper co neu normalized data)
scaler = StandardScaler()
X_norm = scaler.fit_transform(X_clean).astype(np.float32)

# Fit PCA tren mau sach sau loai outlier
pca_full = PCA(n_components=9, random_state=42)
pca_full.fit(X_norm)

# Bảng 8 paper: gán PCi -> vi, % = phương sai giải thích của PCi
pca_table_df = pd.DataFrame({
    "Attribute No.": np.arange(1, len(feature_cols) + 1),
    "feature_col": feature_cols,
    "Feature": [feature_name_map[c] for c in feature_cols],
    "Percentage of Importance": (100.0 * pca_full.explained_variance_ratio_).round(4),
})

print("Table 8. PCA Results (paper mapping: v1..v9 <-> PC1..PC9)")
display(pca_table_df[["Attribute No.", "Feature", "Percentage of Importance"]])

# Paper chọn 3 feature cố định theo Bảng 8: v1, v2, v3
PAPER_TOP3_FEATURES = [
    "clump_thickness",           # v1
    "uniformity_of_cell_size",   # v2
    "uniformity_of_cell_shape",  # v3
]
selected_feature_names = PAPER_TOP3_FEATURES
selected_idx = [feature_cols.index(c) for c in selected_feature_names]
X_selected = X_norm[:, selected_idx].astype(np.float32)

paper_top3_pct = pca_table_df.set_index("feature_col").loc[selected_feature_names, "Percentage of Importance"]

print("\n3 features for ANFIS (paper v1, v2, v3):", selected_feature_names)
print(
    "Cumulative PCA importance of selected features (%):",
    round(float(paper_top3_pct.sum()), 4),
)
print("PC1 explained variance ratio:", round(float(pca_full.explained_variance_ratio_[0]), 4))

# Paper ghi 200/263/200 => tong 663 mau.
# Loai mau theo chat luong tren tap feature da chon (cung hang voi X_selected / y_clean).
n_drop = len(X_selected) - 663

centroid_benign = X_selected[y_clean == 0].mean(axis=0)
centroid_malignant = X_selected[y_clean == 1].mean(axis=0)
dist_to_benign = np.linalg.norm(X_selected - centroid_benign, axis=1)
dist_to_malignant = np.linalg.norm(X_selected - centroid_malignant, axis=1)
own_dist = np.where(y_clean == 0, dist_to_benign, dist_to_malignant)
other_dist = np.where(y_clean == 0, dist_to_malignant, dist_to_benign)
quality_score = other_dist - own_dist

drop_idx = np.argsort(quality_score)[:n_drop]
quality_keep_mask = np.ones(len(X_selected), dtype=bool)
quality_keep_mask[drop_idx] = False

indices = np.where(quality_keep_mask)[0]
X_663 = X_selected[indices]
y_663 = y_clean[indices]

quality_dropped_df = df_clean.iloc[drop_idx][["sample_code_number", "class"]].copy()
quality_dropped_df["quality_score"] = quality_score[drop_idx]
quality_dropped_df = quality_dropped_df.sort_values("quality_score", ascending=True).reset_index(drop=True)

# Split dung kich thuoc paper va giu ty le class on dinh (stratified)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_663, y_663, train_size=200, random_state=42, stratify=y_663,
)
X_check, X_test, y_check, y_test = train_test_split(
    X_temp, y_temp, train_size=263, random_state=42, stratify=y_temp,
)

print("\nSplit theo paper (train/check/test):", X_train.shape, X_check.shape, X_test.shape)
print("So mau bi bo de bam paper:", int(len(X_selected) - len(X_663)))
print("Class sau drop 663 -> benign:", int((y_663 == 0).sum()), "malignant:", int((y_663 == 1).sum()))
print("Ty le malignant sau drop:", round(float((y_663 == 1).mean()), 4))



Outlier Euclidean (mean + 2.5*std): 18 mau
Nguong outlier: 16.5121
Con lai sau loai outlier: 665
Table 8. PCA Results (paper mapping: v1..v9 <-> PC1..PC9)


,Attribute No.,Feature,Percentage of Importance
0,1,Clump Thickness,62.536301
1,2,Uniformity of Cell Size,9.301200
2,3,Uniformity of Cell Shape,6.227000
3,4,Marginal Adhesion,5.730000
4,5,Single Epithelial Cell Size,4.661500
5,6,Bare Nuclei,3.737100
6,7,Bland Chromatin,3.498900
7,8,Normal Nucleoli,3.184500
8,9,Mitoses,1.123500



3 features for ANFIS (paper v1, v2, v3): ['clump_thickness', 'uniformity_of_cell_size', 'uniformity_of_cell_shape']
Cumulative PCA importance of selected features (%): 78.0645
PC1 explained variance ratio: 0.6254

Split theo paper (train/check/test): (200, 3) (263, 3) (200, 3)
So mau bi bo de bam paper: 2
Class sau drop 663 -> benign: 442 malignant: 221
Ty le malignant sau drop: 0.3333


In [ ]:
# 3) Dinh nghia he mo FS() + 27 luat grid + train ANFIS (hybrid)
# Gaussian MF (3/input) + grid partitioning 3^3 = 27 luat, THEN = lanh tinh / ac tinh

import itertools

epochs = 300

# Ten bien trong FS (khong dau cach — parser scikit-anfis khong chap nhan ten co khoang trang)
FS_VAR_NAMES = {
    "clump_thickness": "ClumpThickness",
    "uniformity_of_cell_size": "CellSize",
    "uniformity_of_cell_shape": "CellShape",
}
FS_DISPLAY_NAMES = {v: feature_name_map[k] for k, v in FS_VAR_NAMES.items()}
FS_VAR_LIST = [FS_VAR_NAMES[c] for c in selected_feature_names]
LINGUISTIC_TERMS = ("low", "medium", "high")

# --- Ham thanh vien Gaussian: low / medium / high (khoi tao tu tap train) ---
fs = FS()
mf_init_log = []

for feat_col in selected_feature_names:
    fs_var = FS_VAR_NAMES[feat_col]
    col = X_train[:, selected_feature_names.index(feat_col)]
    col_min, col_max = float(col.min()), float(col.max())
    centers = np.linspace(col_min, col_max, 3).tolist()
    sigma = max((col_max - col_min) / 3.0, 1e-3)

    mf_low = GaussianFuzzySet(mu=centers[0], sigma=sigma, term="low")
    mf_med = GaussianFuzzySet(mu=centers[1], sigma=sigma, term="medium")
    mf_high = GaussianFuzzySet(mu=centers[2], sigma=sigma, term="high")
    fs.add_linguistic_variable(
        fs_var,
        LinguisticVariable([mf_low, mf_med, mf_high], concept=FS_DISPLAY_NAMES[fs_var]),
    )
    mf_init_log.append({
        "var": FS_DISPLAY_NAMES[fs_var],
        "mu": [round(c, 4) for c in centers],
        "sigma": round(sigma, 4),
    })

# --- Dau ra phan loai: crisp 0 = lanh tinh, 1 = ac tinh ---
fs.set_crisp_output_value("benign", 0)
fs.set_crisp_output_value("malignant", 1)


def expert_diagnosis(clump_term, size_term, shape_term):
    """Chuyen gia gan nhan: dac trung thap -> lanh tinh, cao -> ac tinh."""
    score = {"low": 0, "medium": 1, "high": 2}
    v = [score[clump_term], score[size_term], score[shape_term]]
    high_count = sum(x == 2 for x in v)
    if high_count >= 2:
        return "malignant"
    if sum(v) <= 2:
        return "benign"
    if v[0] == 2:  # Clump Thickness cao la dau hieu manh
        return "malignant"
    if sum(v) >= 4:
        return "malignant"
    return "benign"


# --- 27 luat: grid partitioning (AND tat ca to hop low/medium/high) ---
EXPERT_RULES = []
for ct, cs, csh in itertools.product(LINGUISTIC_TERMS, repeat=3):
    diagnosis = expert_diagnosis(ct, cs, csh)
    EXPERT_RULES.append(
        f"IF (ClumpThickness IS {ct}) AND (CellSize IS {cs}) AND (CellShape IS {csh}) "
        f"THEN (Diagnosis IS {diagnosis})"
    )
fs.add_rules(EXPERT_RULES)

print(f"He mo FS: {len(EXPERT_RULES)} luat (grid 3^3 = 27)")
print("Khoi tao Gaussian MF:")
for item in mf_init_log:
    print(f"  {item['var']}: mu={item['mu']}, sigma={item['sigma']}")
print("Vi du 3 luat dau:")
for r in EXPERT_RULES[:3]:
    print(" ", r)

# --- Tao model ANFIS tu FS (zerotype=True => THEN chi la hang so benign/malignant) ---
model = scikit_anfis(
    fs,
    description="WBCD_Gaussian27_ExpertRules_ANFIS",
    epoch=epochs,
    hybrid=True,
    label="c",
    zerotype=True,
    
)

# --- Train hybrid dung chuan ANFIS (on dinh) ---
# Buoc 1 (LSE): cap nhat consequent theo MF hien tai
# Buoc 2 (GD):  cap nhat chi MF (mu/sigma), consequent co dinh trong buoc nay
#
# Adam lr=0.05 + inner_steps=5 de gay vo sigma Gaussian -> RMSE nhay dot ngot (vd epoch ~260).
# Dung SGD lr nho, 1 buoc GD/epoch, clip gradient, clamp sigma, early stopping.

learning_rate = 0.1          # paper-style, on dinh hon Adam 0.05
inner_steps = 1
grad_clip_norm = 1.0
sigma_min = 1e-3
early_stop_patience = 50       # dung neu RMSE khong cai thien 50 epoch lien tiep
patience_counter = 0
best_epoch = 0

optimizer = torch.optim.SGD(model.layer["fuzzify"].parameters(), lr=learning_rate)
criterion = RMSELoss()

X_train_t = torch.from_numpy(X_train).float()
y_train_t = torch.from_numpy(y_train).float().unsqueeze(-1)


def clamp_gaussian_sigma(model, min_sigma=sigma_min):
    """Giu sigma > 0 de tranh MF Gaussian bung no."""
    with torch.no_grad():
        for fuzzify_var in model.layer["fuzzify"].varmfs.values():
            for mf in fuzzify_var.mfdefs.values():
                if hasattr(mf, "sigma"):
                    mf.sigma.data.clamp_(min=min_sigma)


history = []
best_rmse = float("inf")
best_ckpt = Path("tmp.pkl")

model.train()

for ep in range(1, epochs + 1):
    # Buoc 1: LSE cap nhat consequent — khong tinh gradient
    with torch.no_grad():
        model.is_training = True
        model(X_train_t, y_train_t)

    # Buoc 2: GD chi cap nhat MF (premise)
    model.is_training = False
    for _ in range(inner_steps):
        y_pred = model(X_train_t, y_train_t)
        optimizer.zero_grad()
        loss = criterion(y_pred, y_train_t)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.layer["fuzzify"].parameters(), grad_clip_norm)
        optimizer.step()
        clamp_gaussian_sigma(model)

    mse = torch.nn.functional.mse_loss(y_pred, y_train_t).item()
    rmse = float(np.sqrt(mse))
    train_acc = float(
        (torch.clamp(torch.round(y_pred), 0, 1) == y_train_t).float().mean().item()
    )

    unstable = (not np.isfinite(rmse)) or (rmse > max(best_rmse * 2.5, 0.35))
    if unstable:
        print(
            f"Epoch {ep:03d}: RMSE={rmse:.4f} bat thuong, rollback best (ep {best_epoch}, rmse {best_rmse:.4f})"
        )
        model.load(str(best_ckpt))
        model.is_training = False
        with torch.no_grad():
            y_pred = model(X_train_t, y_train_t)
        mse = torch.nn.functional.mse_loss(y_pred, y_train_t).item()
        rmse = float(np.sqrt(mse))
        train_acc = float(
            (torch.clamp(torch.round(y_pred), 0, 1) == y_train_t).float().mean().item()
        )
        patience_counter += 1
    elif rmse < best_rmse - 1e-7:
        best_rmse = rmse
        best_epoch = ep
        model.save(str(best_ckpt))
        patience_counter = 0
    else:
        patience_counter += 1

    history.append(
        {
            "epoch": ep,
            "mse": mse,
            "rmse": rmse,
            "loss": float(loss.item()),
            "train_accuracy": train_acc,
            "lr": optimizer.param_groups[0]["lr"],
            "is_best": int(ep == best_epoch),
        }
    )

    if ep % 25 == 0 or ep == 1:
        print(
            f"Epoch {ep:03d}/{epochs} - RMSE: {rmse:.6f} - Train Acc: {train_acc:.4f}"
            f" - Best: {best_rmse:.6f} @ ep {best_epoch}"
        )

    if patience_counter >= early_stop_patience:
        print(f"\nEarly stopping tai epoch {ep} (best epoch {best_epoch}, RMSE {best_rmse:.6f})")
        break

# Load best checkpoint (coeff + MF dong bo)
model.load(str(best_ckpt))
model.eval()
model.is_training = False

loss_log_df = pd.DataFrame(history)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = Path("models")
out_dir.mkdir(exist_ok=True)
loss_log_path = out_dir / f"{stamp}_paperstyle_gaussian27_training_loss_log.csv"
loss_log_df.to_csv(loss_log_path, index=False)

print("\nBest train RMSE:", round(best_rmse, 6), f"@ epoch {best_epoch}")
print("Num rules:", model.num_rules)
print("Loss log saved:", loss_log_path)

 * Detected Sugeno model type
He mo FS: 27 luat (grid 3^3 = 27)
Khoi tao Gaussian MF:
  Clump Thickness: mu=[-1.2045, 0.417, 2.0384], sigma=1.081
  Uniformity of Cell Size: mu=[-0.6809, 0.8781, 2.4372], sigma=1.0394
  Uniformity of Cell Shape: mu=[-0.7237, 0.8793, 2.4824], sigma=1.0687
Vi du 3 luat dau:
  IF (ClumpThickness IS low) AND (CellSize IS low) AND (CellShape IS low) THEN (Diagnosis IS benign)
  IF (ClumpThickness IS low) AND (CellSize IS low) AND (CellShape IS medium) THEN (Diagnosis IS benign)
  IF (ClumpThickness IS low) AND (CellSize IS low) AND (CellShape IS high) THEN (Diagnosis IS benign)
Epoch 001/300 - RMSE: 0.130647 - Train Acc: 0.9850 - Best: 0.130647 @ ep 1
Epoch 025/300 - RMSE: 0.130249 - Train Acc: 0.9850 - Best: 0.130249 @ ep 25
Epoch 050/300 - RMSE: 0.129823 - Train Acc: 0.9800 - Best: 0.129823 @ ep 50
Epoch 075/300 - RMSE: 0.129381 - Train Acc: 0.9800 - Best: 0.129381 @ ep 75
Epoch 100/300 - RMSE: 0.128915 - Train Acc: 0.9800 - Best: 0.128915 @ ep 100
Epoch 12

In [21]:
# 3b) Hien thi ham thanh vien Gaussian + 27 luat mo (THEN = lanh tinh / ac tinh)

TERM_LABELS = {
    "low": "Thấp (low)",
    "medium": "Trung bình (medium)",
    "high": "Cao (high)",
}
DIAGNOSIS_LABELS = {
    "benign": "Lành tính (benign = 0)",
    "malignant": "Ác tính (malignant = 1)",
}


def prettify_antecedent(text: str) -> str:
    out = text
    for fs_var, display in FS_DISPLAY_NAMES.items():
        out = out.replace(fs_var, display)
    for term, label in TERM_LABELS.items():
        out = out.replace(f" IS {term}", f" IS {label}")
    return out


def build_membership_table(fs_obj, display_names):
    rows = []
    for var_name, lv in fs_obj._lvs.items():
        display = display_names.get(var_name, var_name)
        for mf in lv._FSlist:
            mf_type, params = mf.get_params()
            row = {
                "Input": display,
                "Linguistic term": TERM_LABELS.get(mf.get_term(), mf.get_term()),
            }
            if mf_type == "GaussMembFunc":
                mu, sigma = params
                row["mu (center)"] = round(mu, 4)
                row["sigma"] = round(sigma, 4)
            rows.append(row)
    return pd.DataFrame(rows)


def build_fuzzy_rules_table(expert_rules, display_names):
    rows = []
    for i, rule in enumerate(expert_rules, start=1):
        if_part, then_part = rule.split(" THEN ")
        if_part = if_part.replace("IF ", "", 1)
        diagnosis = then_part.split("IS")[-1].strip().strip(")").strip()
        rows.append(
            {
                "Rule": i,
                "IF": prettify_antecedent(if_part),
                "THEN Diagnosis": DIAGNOSIS_LABELS.get(diagnosis, diagnosis),
            }
        )
    return pd.DataFrame(rows)


membership_df = build_membership_table(fs, FS_DISPLAY_NAMES)
fuzzy_rules_df = build_fuzzy_rules_table(EXPERT_RULES, FS_DISPLAY_NAMES)

print("Membership functions (Gaussian, 3 terms / input)")
display(membership_df)

print(f"\nFuzzy rules ({len(fuzzy_rules_df)} rules = 3^3 grid partitioning)")
display(fuzzy_rules_df)

print("\nSample (5 luat dau):")
for r in EXPERT_RULES[:5]:
    print(" ", prettify_antecedent(r.split(" THEN ")[0].replace("IF ", "", 1)),
          "=>", DIAGNOSIS_LABELS.get(r.split("IS")[-1].strip().strip(")").strip(), ""))

Membership functions (Gaussian, 3 terms / input)


,Input,Linguistic term,mu (center),sigma
0,Clump Thickness,Thấp (low),-1.2045,1.0810
1,Clump Thickness,Trung bình (medium),0.4170,1.0810
2,Clump Thickness,Cao (high),2.0384,1.0810
3,Uniformity of Cell Size,Thấp (low),-0.6809,1.0394
4,Uniformity of Cell Size,Trung bình (medium),0.8781,1.0394
5,Uniformity of Cell Size,Cao (high),2.4372,1.0394
6,Uniformity of Cell Shape,Thấp (low),-0.7237,1.0687
7,Uniformity of Cell Shape,Trung bình (medium),0.8793,1.0687
8,Uniformity of Cell Shape,Cao (high),2.4824,1.0687



Fuzzy rules (27 rules = 3^3 grid partitioning)


,Rule,IF,THEN Diagnosis
0,1,(Clump Thickness IS Thấp (low)) AND (Uniformit...,Lành tính (benign = 0)
1,2,(Clump Thickness IS Thấp (low)) AND (Uniformit...,Lành tính (benign = 0)
2,3,(Clump Thickness IS Thấp (low)) AND (Uniformit...,Lành tính (benign = 0)
3,4,(Clump Thickness IS Thấp (low)) AND (Uniformit...,Lành tính (benign = 0)
4,5,(Clump Thickness IS Thấp (low)) AND (Uniformit...,Lành tính (benign = 0)
5,6,(Clump Thickness IS Thấp (low)) AND (Uniformit...,Lành tính (benign = 0)
6,7,(Clump Thickness IS Thấp (low)) AND (Uniformit...,Lành tính (benign = 0)
7,8,(Clump Thickness IS Thấp (low)) AND (Uniformit...,Lành tính (benign = 0)
8,9,(Clump Thickness IS Thấp (low)) AND (Uniformit...,Ác tính (malignant = 1)
9,10,(Clump Thickness IS Trung bình (medium)) AND (...,Lành tính (benign = 0)



Sample (5 luat dau):
  (Clump Thickness IS Thấp (low)) AND (Uniformity of Cell Size IS Thấp (low)) AND (Uniformity of Cell Shape IS Thấp (low)) => Lành tính (benign = 0)
  (Clump Thickness IS Thấp (low)) AND (Uniformity of Cell Size IS Thấp (low)) AND (Uniformity of Cell Shape IS Trung bình (medium)) => Lành tính (benign = 0)
  (Clump Thickness IS Thấp (low)) AND (Uniformity of Cell Size IS Thấp (low)) AND (Uniformity of Cell Shape IS Cao (high)) => Lành tính (benign = 0)
  (Clump Thickness IS Thấp (low)) AND (Uniformity of Cell Size IS Trung bình (medium)) AND (Uniformity of Cell Shape IS Thấp (low)) => Lành tính (benign = 0)
  (Clump Thickness IS Thấp (low)) AND (Uniformity of Cell Size IS Trung bình (medium)) AND (Uniformity of Cell Shape IS Trung bình (medium)) => Lành tính (benign = 0)


In [22]:
# 4) Evaluate tren check/test + tao bang metric de xuat JSON/CSV

def to_binary(pred):
    # scikit-anfis label='c' tra ve gia tri da lam tron, nhung van clip de an toan
    pred = np.asarray(pred).reshape(-1)
    pred = np.clip(np.round(pred), 0, 1).astype(int)
    return pred

# Checking set
y_check_pred_raw = model.predict(X_check)
y_check_pred = to_binary(y_check_pred_raw)

# Test set
y_test_pred_raw = model.predict(X_test)
y_test_pred = to_binary(y_test_pred_raw)


def evaluate_split(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_pred)
    except ValueError:
        auc = np.nan

    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    print(f"\n{name} metrics")
    print("- Accuracy :", round(acc, 4))
    print("- Precision:", round(prec, 4))
    print("- Recall   :", round(rec, 4))
    print("- F1-score :", round(f1, 4))
    print("- ROC-AUC  :", round(auc, 4) if not np.isnan(auc) else "nan")
    print("- Confusion matrix:\n", cm)

    return {
        "split": name,
        "accuracy": float(acc),
        "precision": float(prec),
        "recall": float(rec),
        "f1_score": float(f1),
        "roc_auc": None if np.isnan(auc) else float(auc),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "support": int(len(y_true)),
    }

check_metrics = evaluate_split("CHECK", y_check.astype(int), y_check_pred)
test_metrics = evaluate_split("TEST", y_test.astype(int), y_test_pred)

metrics_df = pd.DataFrame([check_metrics, test_metrics])
metrics_summary = {
    "CHECK": check_metrics,
    "TEST": test_metrics,
}


CHECK metrics
- Accuracy : 0.943
- Precision: 0.9
- Recall   : 0.931
- F1-score : 0.9153
- ROC-AUC  : 0.9399
- Confusion matrix:
 [[167   9]
 [  6  81]]

TEST metrics
- Accuracy : 0.935
- Precision: 0.8971
- Recall   : 0.9104
- F1-score : 0.9037
- ROC-AUC  : 0.9289
- Confusion matrix:
 [[126   7]
 [  6  61]]


In [23]:
# 5) Luu artifact phuc vu bao cao
import json
import pickle

best_model_path = out_dir / f"{stamp}_paperstyle_pca_feature_select_best_model.pkl"
scaler_path = out_dir / f"{stamp}_paperstyle_scaler.pkl"
pca_info_path = out_dir / f"{stamp}_paperstyle_pca_info.pkl"
metrics_json_path = out_dir / f"{stamp}_paperstyle_metrics.json"
metrics_csv_path = out_dir / f"{stamp}_paperstyle_metrics.csv"
fuzzy_rules_csv_path = out_dir / f"{stamp}_paperstyle_fuzzy_rules.csv"
membership_csv_path = out_dir / f"{stamp}_paperstyle_membership_functions.csv"

# Luu model ANFIS
model.save(str(best_model_path))

# Luu scaler & PCA object (de tai lap phan feature selection)
with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)
with open(pca_info_path, "wb") as f:
    pickle.dump(pca_full, f)

# Luu metric sang JSON va CSV de thong ke/bieu do
metrics_payload = {
    "timestamp": stamp,
    "run_tag": "paperstyle_pca_feature_select",
    "metrics": metrics_summary,
}
with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2, ensure_ascii=False)

metrics_df.to_csv(metrics_csv_path, index=False)
fuzzy_rules_df.to_csv(fuzzy_rules_csv_path, index=False)
membership_df.to_csv(membership_csv_path, index=False)

# Luu danh sach mau bi loai theo Euclidean outlier
outlier_drop_csv_path = out_dir / f"{stamp}_paperstyle_outlier_dropped_samples.csv"
quality_drop_csv_path = out_dir / f"{stamp}_paperstyle_quality_dropped_samples.csv"

if "outlier_dropped_df" not in globals():
    outlier_dropped_df = pd.DataFrame(columns=["sample_code_number", "class", "euclidean_distance", "outlier_threshold"])
if "quality_dropped_df" not in globals():
    quality_dropped_df = pd.DataFrame(columns=["sample_code_number", "class", "quality_score"])
if "outlier_mask" not in globals():
    outlier_mask = np.zeros(len(y), dtype=bool)
if "y_all" not in globals():
    y_all = y.astype(int)

outlier_dropped_df.to_csv(outlier_drop_csv_path, index=False)
quality_dropped_df.to_csv(quality_drop_csv_path, index=False)

meta = {
    "timestamp": stamp,
    "dataset": "UCI WBCD original",
    "n_features_raw": int(X.shape[1]),
    "feature_selection_method": "Paper Table 8: PCA on 9 normalized features; fixed v1/v2/v3",
    "paper_top3_cumulative_pca_importance_pct": float(paper_top3_pct.sum()),
    "pca_table8": pca_table_df[["Attribute No.", "Feature", "Percentage of Importance"]].to_dict(orient="records"),
    "selected_features": selected_feature_names,
    "pca_explained_variance_ratio": pca_full.explained_variance_ratio_.tolist(),
    "split_mode": "paper-like 200/263/200",
    "outlier_method": "Euclidean distance to data center on 9 raw features; threshold = mean + OUTLIER_STD_MULTIPLIER * std",
    "outlier_std_multiplier": OUTLIER_STD_MULTIPLIER,
    "outlier_threshold": float(outlier_threshold),
    "outliers_removed": int(outlier_mask.sum()),
    "used_samples": int(len(X_663)),
    "outlier_drop_class_distribution": pd.Series(y_all[outlier_mask]).value_counts().to_dict(),
    "quality_dropped_sample_code_numbers": quality_dropped_df["sample_code_number"].astype(int).tolist(),
    "class_distribution_663": {
        "benign": int((y_663 == 0).sum()),
        "malignant": int((y_663 == 1).sum()),
    },
    "train_size": int(len(X_train)),
    "check_size": int(len(X_check)),
    "test_size": int(len(X_test)),
    "epoch": epochs,
    "optimizer": "SGD (premise/MF only)",
    "learning_rate": learning_rate,
    "inner_steps_per_epoch": inner_steps,
    "grad_clip_norm": grad_clip_norm,
    "sigma_min": sigma_min,
    "early_stop_patience": early_stop_patience,
    "best_epoch": int(best_epoch),
    "training_mode": "hybrid 2-step + grad clip + sigma clamp + early stop + rollback",
    "fuzzy_system": "FS() + grid 27 rules + expert THEN benign/malignant",
    "membership_function": "Gaussian (low / medium / high)",
    "membership_per_input": 3,
    "rule_partitioning": "grid 3^3 = 27",
    "mf_init_log": mf_init_log,
    "output_type": "crisp classification: benign=0, malignant=1",
    "num_rules": int(model.num_rules),
    "expert_rules": EXPERT_RULES,
    "best_train_rmse": float(best_rmse),
    "loss_log_path": str(loss_log_path),
    "best_model_path": str(best_model_path),
    "metrics_json_path": str(metrics_json_path),
    "metrics_csv_path": str(metrics_csv_path),
    "fuzzy_rules_csv_path": str(fuzzy_rules_csv_path),
    "membership_csv_path": str(membership_csv_path),
    "num_fuzzy_rules": int(len(fuzzy_rules_df)),
    "quality_drop_csv_path": str(quality_drop_csv_path),
}

meta_path = out_dir / f"{stamp}_paperstyle_meta.json"
pd.Series(meta).to_json(meta_path, indent=2)

print("Saved:")
print("-", best_model_path)
print("-", scaler_path)
print("-", pca_info_path)
print("-", metrics_json_path)
print("-", metrics_csv_path)
print("-", fuzzy_rules_csv_path)
print("-", membership_csv_path)
print("-", loss_log_path)
print("-", quality_drop_csv_path)
print("-", meta_path)

Saved:
- models\20260624_073217_paperstyle_pca_feature_select_best_model.pkl
- models\20260624_073217_paperstyle_scaler.pkl
- models\20260624_073217_paperstyle_pca_info.pkl
- models\20260624_073217_paperstyle_metrics.json
- models\20260624_073217_paperstyle_metrics.csv
- models\20260624_073217_paperstyle_fuzzy_rules.csv
- models\20260624_073217_paperstyle_membership_functions.csv
- models\20260624_073217_paperstyle_gaussian27_training_loss_log.csv
- models\20260624_073217_paperstyle_quality_dropped_samples.csv
- models\20260624_073217_paperstyle_meta.json


# Convert HTML

In [24]:
from datetime import datetime

html_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
html_trial_tag = "paperstyle_pca_feature_select"
html_split_tag = "ANFIS_-_split_200_263_200"
html_output_name = f"{html_timestamp}_{html_trial_tag}_{html_split_tag}.html"

log_dir = Path("nhat-ky")
log_dir.mkdir(parents=True, exist_ok=True)

!jupyter nbconvert --to html anfis_pca_scikit_anfis_training.ipynb --output {html_output_name} --output-dir ./nhat-ky

[NbConvertApp] Converting notebook anfis_pca_scikit_anfis_training.ipynb to html
[NbConvertApp] Writing 409355 bytes to nhat-ky\20260624_073217_paperstyle_pca_feature_select_ANFIS_-_split_200_263_200.html
